In [1]:
#1
import os
os.chdir("/kaggle/working")
if not os.path.isdir("CatVTON"):
    !git clone https://github.com/Zheng-Chong/CatVTON.git
%cd /kaggle/working/CatVTON
!grep -vi '^gradio' requirements.txt > requirements_notebook.txt
!pip install -q -r requirements_notebook.txt
!pip install -q huggingface_hub fvcore iopath yacs pycocotools omegaconf cloudpickle av
!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio


Cloning into 'CatVTON'...
remote: Enumerating objects: 1358, done.
remote: Counting objects: 100% (309/309), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 1358 (delta 169), reused 128 (delta 128), pack-reused 1049 (from 2)
Receiving objects: 100% (1358/1358), 16.73 MiB | 44.60 MiB/s, done.
Resolving deltas: 100% (462/462), done.
/kaggle/working/CatVTON
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
#2
import os, time, io, glob
import torch
from PIL import Image
from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import snapshot_download
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

def find_local_dir(root, folder_name):
    """Tìm folder_name bên trong root, trả về path nếu tồn tại và có file."""
    if not os.path.isdir(root):
        return None
    for m in glob.glob(os.path.join(root, "**", folder_name), recursive=True):
        if os.path.isdir(m) and os.listdir(m):
            return m
    return None

DATASET_ROOT = "/kaggle/input/catvton-weights"    # tên dataset sau khi Add Data ở bước 4
WORKING_ROOT = "/kaggle/working/catvton_weights"  # nơi lưu khi tải mới, để publish dataset sau

# --- CatVTON attn checkpoint ---
catvton_local = find_local_dir(DATASET_ROOT, "CatVTON") or os.path.join(WORKING_ROOT, "CatVTON")
if os.path.isdir(catvton_local) and os.listdir(catvton_local):
    repo_path = catvton_local
    print(f"[cache] Dùng CatVTON weights có sẵn: {repo_path}")
else:
    print("[download] Chưa có cache, tải CatVTON từ HF Hub...")
    repo_path = snapshot_download(repo_id="zhengchong/CatVTON", local_dir=catvton_local)

# --- SD inpainting base checkpoint ---
sd_local = find_local_dir(DATASET_ROOT, "sd-inpainting") or os.path.join(WORKING_ROOT, "sd-inpainting")
if os.path.isdir(sd_local) and os.listdir(sd_local):
    base_ckpt_path = sd_local
    print(f"[cache] Dùng SD-inpainting weights có sẵn: {base_ckpt_path}")
else:
    print("[download] Chưa có cache, tải SD-inpainting từ HF Hub...")
    base_ckpt_path = snapshot_download(repo_id="booksforcharlie/stable-diffusion-inpainting", local_dir=sd_local)

pipeline = CatVTONPipeline(
    base_ckpt=base_ckpt_path,
    attn_ckpt=repo_path,
    attn_ckpt_version="mix",
    weight_dtype=init_weight_dtype("fp16"),
    use_tf32=True,
    device="cuda",
)
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True)
automasker = AutoMasker(densepose_ckpt=os.path.join(repo_path, "DensePose"), schp_ckpt=os.path.join(repo_path, "SCHP"), device="cuda")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


[download] Chưa có cache, tải CatVTON từ HF Hub...


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

[download] Chưa có cache, tải SD-inpainting từ HF Hub...


Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /kaggle/working/catvton_weights/sd-inpainting
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
An error occurred while trying to fetch /kaggle/working/catvton_weights/sd-inpainting: Error no file named diffusion_pytorch_model.safetensors found in directory /kaggle/working/catvton_weights/sd-inpainting.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


In [5]:
#3
import nest_asyncio
import uvicorn
from pyngrok import ngrok
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

@app.post("/api/tryon")
async def tryon_api(person_image: UploadFile = File(...), cloth_image: UploadFile = File(...)):
    person_bytes = await person_image.read()
    cloth_bytes = await cloth_image.read()
    person_img = Image.open(io.BytesIO(person_bytes)).convert("RGB")
    cloth_img = Image.open(io.BytesIO(cloth_bytes)).convert("RGB")
    
    WIDTH, HEIGHT = 768, 1024
    person_img = resize_and_crop(person_img, (WIDTH, HEIGHT))
    cloth_img = resize_and_padding(cloth_img, (WIDTH, HEIGHT))
    
    mask = automasker(person_img, "upper")["mask"]
    mask = mask_processor.blur(mask, blur_factor=9)
    
    result = pipeline(image=person_img, condition_image=cloth_img, mask=mask, num_inference_steps=25, guidance_scale=2.5)[0]
    out_path = os.path.join(OUTPUT_DIR, f"result_{int(time.time())}.png")
    result.save(out_path)
    return FileResponse(out_path, media_type="image/png")

# ĐIỀN STATIC DOMAIN VÀ TOKEN NGROK CỦA BẠN VÀO ĐÂY
ngrok.set_auth_token("3B91485xasWPb3CYuMwP0qb76S7_7vgxKozCzYTYdnDCtaCFY")
public_url = ngrok.connect(8000, domain="cactaceous-tatum-semiadhesively.ngrok-free.dev").public_url
print(f"API SẴN SÀNG TẠI: {public_url}/api/tryon")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

API SẴN SÀNG TẠI: https://cactaceous-tatum-semiadhesively.ngrok-free.dev/api/tryon


INFO:     Started server process [108]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
100%|██████████| 25/25 [00:55<00:00,  2.21s/it]


INFO:     118.69.7.9:0 - "POST /api/tryon HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [108]
